# Core 05 - System

Objetivo: usar `toolkit.system` como owner de runtime, registry de Tools, Skills, Agents e inspeccion estatica.

**Lugar en el modelo:** `System` posee registries y conecta una o más unidades de cómputo mediante un plan de ejecución externo a cada Agent.

**Evidencia exigida:** Tools, ToolSet, Skill y Agent deben compartir ownership; `system.compile()` y `system.run()` deben ejecutar el plan secuencial observable.

**Límite de la evidencia:** el plan actual es ejecución, no todavía un álgebra composicional completa; esa extensión puede añadirse sin cambiar la frontera Agent/System.

## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_DEMO_SYMBOL | system | Entrada del usuario para el agente registrado. |
| runtime | python-runtime | Mantener la demostracion reproducible. |
| inspect | sin ejecucion | Validar el registro y sus efectos laterales. |

In [ ]:
import os

import agentic_systems as toolkit

SYMBOL = os.getenv("AGENTIC_SYSTEMS_DEMO_SYMBOL", "system")
runtime = toolkit.runtime(provider="python-runtime")
system = toolkit.system(runtime=runtime)

## 1) Registrar Tool y crear un `ToolSet`

`toolkit` sigue siendo el alias de toda la caja de herramientas. `ToolSet` es el objeto concreto que agrupa Tools con nombres como `public_api.package_version`; no contiene Agents, Environments ni Evals.

In [ ]:
@system.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

public_api_tools = toolkit.toolset(system, "public_api")

@public_api_tools.tool
def package_version() -> dict:
    return {"package_version": toolkit.__version__}

assert isinstance(public_api_tools, toolkit.ToolSet)
assert public_api_tools.tool_names == ("public_api.package_version",)

toolkit.show_json({
    "public_tool_names": list(system.public_tool_names),
    "runtime_tool_names": list(system.tool_names),
    "toolset": {"name": public_api_tools.name, "tool_names": list(public_api_tools.tool_names)},
}, title="System registry")

## 2) Registrar Skill reusable

In [ ]:
inspection_skill = toolkit.skill(
    name="system_public_api_inspection",
    description="Capacidad registrada por el System.",
    tools=[system.public_tools["inspect_public_api"]],
    prompts={"instructions": "Inspecciona simbolos con evidencia."},
    contracts={"default": toolkit.AgentContract(must_call=["inspect_public_api"]).model_dump(mode="json")},
)
system.skill(inspection_skill)
toolkit.show_json({
    "skill_names": list(system.skill_names),
    "runtime_skills": [skill.info() for skill in system.runtime_skills],
}, title="System skills")

## 3) Crear un Agent y ejecutar el plan del System

La ejecución directa del Agent prueba la unidad. `system.compile()` hace visible el plan externo y `system.run()` ejecuta todas las unidades registradas en ese orden.

In [ ]:
agent = system.agent(
    name="system_public_api_agent",
    instructions=inspection_skill.instructions,
    skills=[inspection_skill],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
)
request = {"tool": "inspect_public_api", "input": {"symbol": SYMBOL}}
agent_result = agent.run(request, mode="eval")
assert agent_result.ok, agent_result.errors

compiled = system.compile(name="public_api_system")
compiled_inspection = compiled.inspect()
assert compiled_inspection == {
    "name": "public_api_system",
    "execution_plan": "sequential",
    "unit_count": 1,
}
system_result = system.run(request, mode="eval")
assert system_result.ok, system_result.errors
assert len(system_result.children) == 1

toolkit.human_result(agent_result, title="Direct Agent RunResult", show_lineage=True)
toolkit.human_result(system_result, title="System RunResult", show_lineage=True)
toolkit.show_json(compiled_inspection, title="System execution plan")

## 4) Inspeccionar sin ejecutar componentes

`system.inspect()` es estatico. `toolkit.show` selecciona la explicacion humana de `InspectReport`.

In [ ]:
inspection = system.inspect()
inspection.raise_if_errors()
toolkit.show_json(inspection.to_dict(), title="System inspection structured")
toolkit.show(inspection, title="System inspection human")

assert inspection["side_effects"] == {"models_executed": 0, "tools_executed": 0}

## 5) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.runtime", "toolkit.system", "system.tool", "toolkit.toolset", "toolkit.ToolSet",
    "ToolSet.tool", "ToolSet.tool_names", "toolkit.skill", "system.skill",
    "system.agent", "agent.run", "system.compile", "CompiledSystem.inspect", "system.run",
    "toolkit.human_result", "system.inspect",
    "InspectReport.to_dict", "toolkit.show", "toolkit.show_json",
    "toolkit.AgentContract",
]
toolkit.show_json(api_coverage, title="System API coverage")

## Resultado e interpretacion

Registry, Skill y Agent comparten el mismo System. La inspeccion reporta cero side effects propios.